# Microscope Model Runtime From Design Curves

This notebook loads `microscope_design_curves.npz`, selects `(spot_index, system_magnification)`,
builds one active `MicroscopeModel` mode with signed ObjPost/PL1 compensation currents,
keeps `Cmini` explicit in the chain, and propagates a single Gaussian beam.


In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from temgym_core import LensConfig, MicroscopeModel, OperatingMode
from temgym_core.components import Lens, Detector, Plane, ElectromagneticLens
from temgym_core.ray import Ray
from temgym_core.source import make_waist_divergence_rays
from temgym_core.gaussian import make_gaussian
from temgym_core.run import run_to_end
from temgym_core.evaluate import evaluate_gaussians_for
from temgym_core.plotting import plot_model, legacy_beam_plot_params


In [2]:
curve_path_candidates = (
    Path('data/microscope_design_curves.npz'),
    Path('examples/microscope_models/data/microscope_design_curves.npz'),
    Path('examples/microscope_models/examples/microscope_models/data/microscope_design_curves.npz'),
)
curve_path = next((p for p in curve_path_candidates if p.exists()), None)
if curve_path is None:
    tried = '\n  '.join(str(p) for p in curve_path_candidates)
    raise FileNotFoundError(f'Could not find microscope_design_curves.npz. Tried:\n  {tried}')

pkg = np.load(curve_path, allow_pickle=True)
metadata = json.loads(str(pkg['metadata_json'].item()))

spot_control_values = np.asarray(pkg['spot_control_values'], dtype=float)
spot_norm_currents = np.asarray(pkg['spot_norm_currents'], dtype=float)  # [CL1, CL3]
spot_sample_size_nm = np.asarray(pkg['spot_sample_size_nm'], dtype=float)

mag_control_values = np.asarray(pkg['mag_control_values'], dtype=float)
mag_norm_currents_il = np.asarray(pkg['mag_norm_currents_il'], dtype=float)  # [IL1, IL2, IL3]
mag_norm_currents_comp = np.asarray(pkg['mag_norm_currents_comp'], dtype=float)  # [ObjPost, PL1]
mag_gc_scales_comp = np.asarray(pkg['mag_gc_scales_comp'], dtype=float)  # [ObjPost, PL1]

print('Loaded curve package:', curve_path)
print('  spot grid size:', spot_control_values.size)
print('  mag grid size :', mag_control_values.size)
print('  M_obj, M_proj :', metadata['M_obj'], metadata['M_proj'])


Loaded curve package: examples/microscope_models/data/microscope_design_curves.npz
  spot grid size: 8
  mag grid size : 5
  M_obj, M_proj : 80.0 100.0


In [3]:
# -----------------------
# Runtime user controls
# -----------------------
selected_spot_index = 4.0
selected_system_magnification = 100000.0
selection_policy = 'clip'  # 'clip' or 'strict'

# Gaussian input
beam_waist_m = 10e-9
beam_offset_x_m = 0.0
beam_offset_y_m = 0.0

# Detector sampling
sample_pixel_size_nm = 10.0
sample_shape = (256, 256)
screen_pixel_size_nm = 100.0
screen_shape = (256, 256)


def _resolve_control(value, control_values, policy):
    lo = float(np.min(control_values))
    hi = float(np.max(control_values))
    v = float(value)
    if policy == 'strict' and (v < lo or v > hi):
        raise ValueError(f'Value {v} outside [{lo}, {hi}] under strict policy.')
    return float(np.clip(v, lo, hi))


def _interp_vector(x, xs, ys):
    ys = np.asarray(ys, dtype=float)
    out = np.empty(ys.shape[1], dtype=float)
    for i in range(ys.shape[1]):
        out[i] = np.interp(float(x), xs, ys[:, i])
    return out


In [4]:
# -----------------------------------------------------------
# Build one active combined mode for a single MicroscopeModel
# -----------------------------------------------------------
spot_eval = _resolve_control(selected_spot_index, spot_control_values, selection_policy)
mag_eval = _resolve_control(selected_system_magnification, mag_control_values, selection_policy)

i_norm_cond = _interp_vector(spot_eval, spot_control_values, spot_norm_currents)      # CL1, CL3
i_norm_il = _interp_vector(mag_eval, mag_control_values, mag_norm_currents_il)         # IL1, IL2, IL3
i_norm_comp = _interp_vector(mag_eval, mag_control_values, mag_norm_currents_comp)     # ObjPost, PL1

gc_scale_comp = _interp_vector(mag_eval, mag_control_values, mag_gc_scales_comp)       # ObjPost, PL1

i_norm_full = np.array([
    i_norm_cond[0],  # CL1
    i_norm_cond[1],  # CL3
    i_norm_il[0],    # IL1
    i_norm_il[1],    # IL2
    i_norm_il[2],    # IL3
    i_norm_comp[0],  # ObjPost
    i_norm_comp[1],  # PL1
], dtype=float)

gc_scale_full = np.array([
    1.0,  # CL1
    1.0,  # CL3
    1.0,  # IL1
    1.0,  # IL2
    1.0,  # IL3
    gc_scale_comp[0],  # ObjPost
    gc_scale_comp[1],  # PL1
], dtype=float)

lens_order = ['CL1', 'CL3', 'ObjPost', 'IL1', 'IL2', 'IL3', 'PL1']
z = {k: float(v) for k, v in metadata['z_positions_m'].items()}
turns = metadata['turns']
gc_base = metadata['Gc_base']
rc_phys = float(metadata['Rc_phys_rad_per_AT'])
voltage = float(metadata['voltage_V'])
full_scale_current = float(metadata['full_scale_current_A'])

sample_distance_from_opl_m = float(metadata.get('sample_distance_from_opl_m', 0.0))
objpost_gap = max(sample_distance_from_opl_m, 1e-6)

z_runtime = dict(z)
z_sample = float(z_runtime['Sample'])
z_il1 = float(z_runtime['IL1'])
z_objpost_candidate = float(z_runtime['ObjPost'])
if z_objpost_candidate <= z_sample:
    z_objpost_candidate = z_sample + objpost_gap
z_runtime['ObjPost'] = min(z_il1 - 1e-6, max(z_sample + 1e-6, z_objpost_candidate))
if z_runtime['ObjPost'] <= z_sample:
    raise ValueError('Unable to place ObjPost after Sample while keeping it before IL1.')

z_objpre_candidate = float(z_runtime.get('ObjPre', z_sample - objpost_gap))
z_cmini = float(z_runtime['Cmini'])
z_runtime['ObjPre'] = min(z_sample - 1e-6, max(z_cmini + 1e-6, z_objpre_candidate))
if not (z_cmini < z_runtime['ObjPre'] < z_sample):
    raise ValueError('Objective Pre field must lie between Cmini and Sample.')

lens_configs = tuple(
    LensConfig(
        name=name,
        z_position=float(z_runtime[name]),
        turns=float(turns[name]),
        Gc=float(gc_base[name]),
        Rc=float(rc_phys),
        Tc=0.0,
    )
    for name in lens_order
)

print('Runtime geometry (m):')
print(
    f"  Cmini={z_runtime['Cmini']:.6f}, ObjPre={z_runtime['ObjPre']:.6f}, "
    f"Sample={z_runtime['Sample']:.6f}, ObjPost={z_runtime['ObjPost']:.6f}"
)

active_mode = OperatingMode(
    control_values=np.array([0.0, 1.0]),
    normalized_currents=np.vstack([i_norm_full, i_norm_full]),
    full_scale_current=full_scale_current,
    allow_signed_currents=True,
    gc_scales=np.vstack([gc_scale_full, gc_scale_full]),
)

model = MicroscopeModel(
    voltage=voltage,
    reference_voltage=voltage,
    lenses=lens_configs,
    modes={'active': active_mode},
)

em_components = model.build_components('active', 0.0)
comp_by_name = {cfg.name: comp for cfg, comp in zip(lens_configs, em_components)}

# NI cancellation diagnostics
ni_il_total = float(
    comp_by_name['IL1'].excitation
    + comp_by_name['IL2'].excitation
    + comp_by_name['IL3'].excitation
)
ni_comp_total = float(comp_by_name['ObjPost'].excitation + comp_by_name['PL1'].excitation)
ni_residual = float(ni_il_total + ni_comp_total)

theta_il = float(rc_phys * ni_il_total)
theta_comp = float(rc_phys * ni_comp_total)
theta_net = float(theta_il + theta_comp)

rows = []
for cfg, comp in zip(lens_configs, em_components):
    rows.append({
        'lens': cfg.name,
        'z_m': float(cfg.z_position),
        'I_norm': float(i_norm_full[lens_order.index(cfg.name)]),
        'I_A': float(comp.current),
        'NI_AT': float(comp.excitation),
        'Gc': float(comp.Gc),
        'focal_m': float(comp.focal_length),
        'rotation_rad': float(comp.rotation_angle),
    })

df_runtime = pd.DataFrame(rows)
with pd.option_context('display.max_columns', None, 'display.float_format', '{:.9e}'.format):
    display(df_runtime)

print('Resolved controls:')
print(f'  spot_index: requested={selected_spot_index}, used={spot_eval}')
print(f'  system_mag: requested={selected_system_magnification}, used={mag_eval}')
print(f'  estimated sample size at spot [nm]: {np.interp(spot_eval, spot_control_values, spot_sample_size_nm):.6f}')
print('NI cancellation diagnostics:')
print(f'  NI_IL_total [AT]:   {ni_il_total:.6f}')
print(f'  NI_comp_total [AT]: {ni_comp_total:.6f}')
print(f'  NI_residual [AT]:   {ni_residual:.6e}')
print(f'  theta_IL [rad]:     {theta_il:.6e}')
print(f'  theta_comp [rad]:   {theta_comp:.6e}')
print(f'  theta_net [rad]:    {theta_net:.6e}')


Runtime geometry (m):
  Cmini=0.300000, ObjPre=0.310000, Sample=0.315000, ObjPost=0.320000


ValueError: normalized_currents entries must be within [-1, 1] when allow_signed_currents=True.

In [ ]:
# -----------------------------------------------------------------------------
# Build explicit ordered propagation chains for diagram + field calculations
# Desired order: Source -> CL1 -> CL3 -> Cmini -> ObjPre -> Sample -> ObjPost
#                -> IL1 -> IL2 -> IL3 -> PL1 -> Detector
# -----------------------------------------------------------------------------
class CL1(ElectromagneticLens):
    pass


class CL3(ElectromagneticLens):
    pass


class ObjectivePostField(ElectromagneticLens):
    pass


class IL1(ElectromagneticLens):
    pass


class IL2(ElectromagneticLens):
    pass


class IL3(ElectromagneticLens):
    pass


class PL1(ElectromagneticLens):
    pass


class CondenserMini(Lens):
    pass


class ObjectivePreField(Plane):
    pass


BaseDetector = Detector


class Sample(BaseDetector):
    pass


class detector(BaseDetector):
    pass


def _clone_em(cls, comp):
    return cls(
        z=float(comp.z),
        turns=float(comp.turns),
        current=float(comp.current),
        Gc=float(comp.Gc),
        Rc=float(comp.Rc),
        Tc=float(comp.Tc),
        x0=float(comp.x0),
        y0=float(comp.y0),
    )


named_em = {
    'CL1': _clone_em(CL1, comp_by_name['CL1']),
    'CL3': _clone_em(CL3, comp_by_name['CL3']),
    'ObjPost': _clone_em(ObjectivePostField, comp_by_name['ObjPost']),
    'IL1': _clone_em(IL1, comp_by_name['IL1']),
    'IL2': _clone_em(IL2, comp_by_name['IL2']),
    'IL3': _clone_em(IL3, comp_by_name['IL3']),
    'PL1': _clone_em(PL1, comp_by_name['PL1']),
}

z_sample = float(z_runtime['Sample'])
z_screen = float(z_runtime['Screen'])

cmini_lens = CondenserMini(
    z=float(z_runtime['Cmini']),
    focal_length=float(metadata['f_cmini_fixed_m']),
)
objective_pre_field = ObjectivePreField(z=float(z_runtime['ObjPre']))

sample_detector = Sample(
    z=z_sample,
    pixel_size=(sample_pixel_size_nm * 1e-9, sample_pixel_size_nm * 1e-9),
    shape=sample_shape,
)
screen_detector = detector(
    z=z_screen,
    pixel_size=(screen_pixel_size_nm * 1e-9, screen_pixel_size_nm * 1e-9),
    shape=screen_shape,
)

diagram_chain = [
    ('Source', float(z_runtime['Source']), None),
    ('CL1', float(named_em['CL1'].z), named_em['CL1']),
    ('CL3', float(named_em['CL3'].z), named_em['CL3']),
    ('Condenser Mini', float(cmini_lens.z), cmini_lens),
    ('Objective Pre field', float(objective_pre_field.z), objective_pre_field),
    ('Sample', float(sample_detector.z), sample_detector),
    ('Objective Post field', float(named_em['ObjPost'].z), named_em['ObjPost']),
    ('IL1', float(named_em['IL1'].z), named_em['IL1']),
    ('IL2', float(named_em['IL2'].z), named_em['IL2']),
    ('IL3', float(named_em['IL3'].z), named_em['IL3']),
    ('PL1', float(named_em['PL1'].z), named_em['PL1']),
    ('detector', float(screen_detector.z), screen_detector),
]

diagram_chain_z = np.asarray([z_val for _, z_val, _ in diagram_chain], dtype=float)
if np.any(np.diff(diagram_chain_z) <= 0.0):
    raise ValueError('Diagram component z positions must be strictly increasing.')

components_sample = tuple(
    comp
    for name, _, comp in diagram_chain
    if comp is not None and name in {'CL1', 'CL3', 'Condenser Mini', 'Objective Pre field', 'Sample'}
)
components_screen = tuple(comp for _, _, comp in diagram_chain if comp is not None)

beam_in = make_gaussian(
    x=beam_offset_x_m,
    y=beam_offset_y_m,
    dx=0.0,
    dy=0.0,
    z=float(z_runtime['Source']),
    voltage=voltage,
    waist_x=beam_waist_m,
    waist_y=beam_waist_m,
)

beam_at_sample = run_to_end(beam_in, components_sample)
beam_at_screen = run_to_end(beam_in, components_screen)

field_sample = np.asarray(evaluate_gaussians_for(beam_at_sample, sample_detector))
field_screen = np.asarray(evaluate_gaussians_for(beam_at_screen, screen_detector))

int_sample = np.abs(field_sample) ** 2
int_screen = np.abs(field_screen) ** 2

print('Diagram order:')
print('  ' + ' -> '.join(name for name, _, _ in diagram_chain))
print('Propagation done:')
print('  sample intensity max:', float(np.max(int_sample)))
print('  screen intensity max:', float(np.max(int_screen)))


In [ ]:
# ----------------------------
# Side-view + image diagnostics
# ----------------------------
source_basis = make_waist_divergence_rays(
    waist=beam_waist_m,
    voltage=voltage,
    z=float(z_runtime['Source']),
)
center_ray = Ray(
    x=np.asarray([0.0], dtype=float),
    y=np.asarray([0.0], dtype=float),
    dx=np.asarray([0.0], dtype=float),
    dy=np.asarray([0.0], dtype=float),
    z=np.asarray([float(z_runtime['Source'])], dtype=float),
    pathlength=np.asarray([0.0], dtype=float),
)

fig, ax = plot_model(
    tuple(components_screen),
    rays=center_ray,
    solution_rays=source_basis,
    include_input_rays=True,
    plot_params=legacy_beam_plot_params(),
)
ax.set_title('Microscope chain: Source -> CL1 -> CL3 -> Cmini -> ObjPre -> Sample -> ObjPost -> IL1 -> IL2 -> IL3 -> PL1 -> Detector')
plt.show()

sx, sy = sample_detector.coords_1d
sx_nm = np.asarray(sx) * 1e9
sy_nm = np.asarray(sy) * 1e9
sample_extent = (float(np.min(sx_nm)), float(np.max(sx_nm)), float(np.min(sy_nm)), float(np.max(sy_nm)))

kx, ky = screen_detector.coords_1d
kx_nm = np.asarray(kx) * 1e9
ky_nm = np.asarray(ky) * 1e9
screen_extent = (float(np.min(kx_nm)), float(np.max(kx_nm)), float(np.min(ky_nm)), float(np.max(ky_nm)))

fig, axs = plt.subplots(1, 2, figsize=(12, 5), constrained_layout=True)

im0 = axs[0].imshow(int_sample, origin='lower', extent=sample_extent, cmap='inferno')
axs[0].set_title('Sample plane intensity')
axs[0].set_xlabel('x [nm]')
axs[0].set_ylabel('y [nm]')
plt.colorbar(im0, ax=axs[0], fraction=0.046, pad=0.04)

im1 = axs[1].imshow(int_screen, origin='lower', extent=screen_extent, cmap='viridis')
axs[1].set_title('Screen plane intensity')
axs[1].set_xlabel('x [nm]')
axs[1].set_ylabel('y [nm]')
plt.colorbar(im1, ax=axs[1], fraction=0.046, pad=0.04)

plt.show()
